# 종로구 주차장 데이터 정제 · 통합 노트북

**SKN35 1st 4Team** · 담당: 승희

공영 · 민영 · 부설 주차장 데이터를 모아 하나의 테이블로 만들고,
추천 알고리즘까지 검증하는 전 과정을 단계별로 확인합니다.

| 단계 | 내용 |
|---|---|
| 1 | 서울시 공영주차장 원본 탐색 |
| 2 | 종로구 필터링 |
| 3 | 노상주차장 중복 좌표 병합 |
| 4 | 컬럼 매핑 · 타입 정리 |
| 5 | 결측치 진단 |
| 6 | 전국주차장정보표준데이터 (민영 · 부설) |
| 7 | 다중 소스 통합 + 중복 제거 |
| 8 | 예상 요금 · 추천 점수 계산 |
| 9 | 최종 저장 |


## 0. 준비

프로젝트 루트에서 실행해야 `common/`, `collectors/` 모듈을 import할 수 있습니다.

In [25]:
import sys
from pathlib import Path

# 노트북이 notebooks/ 안에 있어도 프로젝트 루트를 찾아 sys.path에 추가
ROOT = Path.cwd()
while not (ROOT / "common").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("프로젝트 루트:", ROOT)

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

프로젝트 루트: /Users/lsh/Library/Application Support/Claude/local-agent-mode-sessions/afaac2f5-30a2-4216-a837-eb123ec06b0c/02bcc11c-9e38-4f9b-92ea-d2845619930d/local_ecbc54ba-1f5e-4442-a756-62b43843cf51/outputs/team-project-scaffold-v7


## 1. 서울시 공영주차장 원본 탐색

데이터셋: [서울시 공영주차장 안내 정보 (OA-13122)](https://data.seoul.go.kr/dataList/OA-13122/S/1/datasetView.do)

인코딩이 EUC-KR이라 그냥 읽으면 깨집니다. 수집 스크립트의 `load_raw()`가 이를 처리합니다.

In [26]:
from collectors.public_parking_api import load_raw

raw = load_raw(str(ROOT / "data/raw/seoul_parking.csv"))
print(f"전체 행수: {len(raw):,}")
print(f"컬럼 수: {len(raw.columns)}")
raw.head(3)

전체 행수: 2,189
컬럼 수: 39


,주차장코드,주차장명,주소,주차장 종류,주차장 종류명,운영구분,운영구분명,전화번호,주차현황 정보 제공여부,주차현황 정보 제공여부명,총 주차면,유무료구분,유무료구분명,야간무료개방여부,야간무료개방여부명,평일 운영 시작시각(HHMM),평일 운영 종료시각(HHMM),주말 운영 시작시각(HHMM),주말 운영 종료시각(HHMM),공휴일 운영 시작시각(HHMM),공휴일 운영 종료시각(HHMM),최종데이터 동기화 시간,"토요일 유,무료 구분","토요일 유,무료 구분명","공휴일 유,무료 구분","공휴일 유,무료 구분명",월 정기권 금액,노상 주차장 관리그룹번호,기본 주차 요금,기본 주차 시간(분 단위),추가 단위 요금,추가 단위 시간(분 단위),버스 기본 주차 요금,버스 기본 주차 시간(분 단위),버스 추가 단위 시간(분 단위),버스 추가 단위 요금,일 최대 요금,위도,경도
0,1013181,마장동(건물) 공영주차장(구),성동구 마장동 463-2,NW,노외 주차장,1,시간제 주차장,02-2204-7970,1,현재~20분이내 연계데이터 존재(현재 주차대수 표현),52,Y,유료,N,야간 미개방,0,2400,0.0,2400.0,0.0,2400.0,2023-06-28 13:25:08,N,무료,N,무료,20000.0,NaN,50.0,5.0,NaN,NaN,0.0,0.0,0.0,0.0,NaN,NaN,NaN
1,1025695,영등포여고 공영(구),영등포구 신길동 184-3,NW,노외 주차장,1,시간제 주차장,02-2677-1401,1,현재~20분이내 연계데이터 존재(현재 주차대수 표현),98,Y,유료,N,야간 미개방,0,2400,0.0,2400.0,0.0,2400.0,2023-06-30 15:53:00,N,무료,N,무료,65000.0,NaN,50.0,5.0,50.0,5.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN
2,1025696,당산근린공원 공영(구),영등포구 당산동3가 385-0,NW,노외 주차장,1,시간제 주차장,02-2677-1401,0,미연계중,190,Y,유료,N,야간 미개방,0,2400,0.0,2400.0,0.0,2400.0,2023-06-30 15:22:17,N,무료,N,무료,100000.0,NaN,150.0,5.0,150.0,5.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN


In [27]:
# 어떤 컬럼이 있는지 전체 확인
list(raw.columns)

['주차장코드',
 '주차장명',
 '주소',
 '주차장 종류',
 '주차장 종류명',
 '운영구분',
 '운영구분명',
 '전화번호',
 '주차현황 정보 제공여부',
 '주차현황 정보 제공여부명',
 '총 주차면',
 '유무료구분',
 '유무료구분명',
 '야간무료개방여부',
 '야간무료개방여부명',
 '평일 운영 시작시각(HHMM)',
 '평일 운영 종료시각(HHMM)',
 '주말 운영 시작시각(HHMM)',
 '주말 운영 종료시각(HHMM)',
 '공휴일 운영 시작시각(HHMM)',
 '공휴일 운영 종료시각(HHMM)',
 '최종데이터 동기화 시간',
 '토요일 유,무료 구분',
 '토요일 유,무료 구분명',
 '공휴일 유,무료 구분',
 '공휴일 유,무료 구분명',
 '월 정기권 금액',
 '노상 주차장 관리그룹번호',
 '기본 주차 요금',
 '기본 주차 시간(분 단위)',
 '추가 단위 요금',
 '추가 단위 시간(분 단위)',
 '버스 기본 주차 요금',
 '버스 기본 주차 시간(분 단위)',
 '버스 추가 단위 시간(분 단위)',
 '버스 추가 단위 요금',
 '일 최대 요금',
 '위도',
 '경도']

### 자치구별 분포 확인

원본에는 `구` 컬럼이 따로 없고 **주소 문자열 안에** 자치구명이 들어있습니다.

In [28]:
gu = raw["주소"].astype(str).str.split().str[0]
gu.value_counts().head(10)

주소
중구      468
영등포구    271
종로구     210
강남구     153
성동구     131
구로구     109
관악구      84
마포구      82
금천구      79
송파구      67
Name: count, dtype: int64

## 2. 종로구 필터링

주소에 '종로구'가 포함된 행만 남깁니다.

In [29]:
from collectors.public_parking_api import filter_district

jongno_raw = filter_district(raw, "종로구")
print(f"종로구 행수:      {len(jongno_raw):,}")
print(f"고유 주차장 수:   {jongno_raw['주차장코드'].nunique():,}")
print()
print("→ 행수와 주차장 수가 다릅니다. 왜일까요?")

종로구 행수:      210
고유 주차장 수:   43

→ 행수와 주차장 수가 다릅니다. 왜일까요?


## 3. 노상주차장 중복 좌표 병합 ⚠️ 핵심 처리

**노상 주차구역**은 길을 따라 늘어서 있어서, 하나의 주차장코드가
**여러 좌표점으로 쪼개져 여러 행**에 저장됩니다.

이걸 그대로 지도에 찍으면 같은 주차장에 마커가 수십 개 찍힙니다.
→ 주차장코드 기준으로 묶고 좌표는 **평균(중심점)** 을 사용합니다.

In [30]:
# 실제로 중복이 심한 주차장 확인
dup_counts = jongno_raw["주차장코드"].value_counts()
print("한 주차장이 몇 개 행으로 쪼개져 있나 (상위 5개):")
print(dup_counts.head())

worst = dup_counts.index[0]
print(f"\n예시 - 주차장코드 {worst} 의 좌표들:")
jongno_raw[jongno_raw["주차장코드"] == worst][["주차장명", "위도", "경도"]].head(8)

한 주차장이 몇 개 행으로 쪼개져 있나 (상위 5개):
주차장코드
1451364    39
1452441    28
1452019    25
1210657    19
1452721    12
Name: count, dtype: int64

예시 - 주차장코드 1451364 의 좌표들:


,주차장명,위도,경도
423,청계7가 공영주차장(시),37.570236,127.016751
424,청계7가 공영주차장(시),37.572159,127.021368
425,청계7가 공영주차장(시),37.570316,127.016985
426,청계7가 공영주차장(시),37.571698,127.019773
427,청계7가 공영주차장(시),37.572162,127.022410
428,청계7가 공영주차장(시),37.570344,127.017059
429,청계7가 공영주차장(시),37.571395,127.019184
430,청계7가 공영주차장(시),37.572104,127.022810


In [31]:
from collectors.public_parking_api import dedupe_by_parking_lot

merged = dedupe_by_parking_lot(jongno_raw)
print(f"병합 전: {len(jongno_raw):,}행")
print(f"병합 후: {len(merged):,}행  (주차장당 1행)")

병합 전: 210행
병합 후: 43행  (주차장당 1행)


## 4. 컬럼 매핑 · 타입 정리

한글 컬럼명을 DB 테이블(`PARKING_LOT`) 스키마의 영문 컬럼으로 바꾸고,
요금·시간 컬럼을 숫자형으로 변환합니다.

In [32]:
from collectors.public_parking_api import to_table_schema, COLUMN_MAP

# 매핑 규칙 확인
pd.DataFrame(COLUMN_MAP.items(), columns=["원본 컬럼", "테이블 컬럼"]).head(12)

,원본 컬럼,테이블 컬럼
0,주차장코드,ext_id
1,주차장명,parking_name
2,주소,address
3,주차장 종류명,lot_type
4,위도,latitude
5,경도,longitude
6,총 주차면,capacity
7,유무료구분명,pay_type
8,기본 주차 요금,base_fee
9,기본 주차 시간(분 단위),base_time


In [33]:
public = to_table_schema(merged)
print(f"{len(public)}행 × {len(public.columns)}컬럼")
public.head(5)

43행 × 23컬럼


,ext_id,parking_name,address,lot_type,latitude,longitude,capacity,pay_type,base_fee,base_time,add_fee,add_time,day_max_fee,month_fee,weekday_start,weekday_end,weekend_start,weekend_end,holiday_start,holiday_end,tel,lot_category,source
0,171721,세종로 공영주차장(시),종로구 세종로 80-1,노외 주차장,37.573403,126.975884,1260,유료,430,5,430,5,<NA>,176000,0,2400,0,2400,0,2400,02-2290-6566,공영,seoul_public
1,171730,종묘주차장 공영주차장(시),종로구 훈정동 2-0,노외 주차장,37.571504,126.994969,1312,유료,400,5,400,5,<NA>,175000,0,2400,0,2400,0,2400,02-2290-6166,공영,seoul_public
2,1208601,청계2(북2) 공영주차장(시),종로구 관수동 91-4,노상 주차장,37.568868,126.996643,1,유료,750,5,750,5,0,0,900,1900,900,1500,0,0,None,공영,seoul_public
3,1210657,청계3(북1) 공영주차장(시),종로구 예지동 140-1,노상 주차장,37.569436,126.999902,1,유료,750,5,750,5,0,0,900,1900,900,1500,0,0,None,공영,seoul_public
4,1369476,탑골공원 관광버스전용 주차장(시),종로구 종로2가 38-4,노상 주차장,37.571305,126.987764,1,무료,0,0,0,0,0,0,0,0,0,0,0,0,02)2290-6449,공영,seoul_public


In [34]:
# 타입이 제대로 잡혔는지 확인 (Int64 = 결측 허용 정수형)
public.dtypes

ext_id             int64
parking_name      object
address           object
lot_type          object
latitude         float64
longitude        float64
capacity           Int64
pay_type          object
base_fee           Int64
base_time          Int64
add_fee            Int64
add_time           Int64
day_max_fee        Int64
month_fee          Int64
weekday_start      Int64
weekday_end        Int64
weekend_start      Int64
weekend_end        Int64
holiday_start      Int64
holiday_end        Int64
tel               object
lot_category      object
source            object
dtype: object

## 5. 결측치 진단

**좌표가 없으면 지도에 못 찍습니다.** 몇 곳이나 되는지 확인합니다.

In [35]:
missing = public.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("결측치가 있는 컬럼:")
print(missing.to_string())

no_coord = public[["latitude", "longitude"]].isna().any(axis=1).sum()
print(f"\n좌표 없음: {no_coord}곳 / 전체 {len(public)}곳")
print(f"지도 표시 가능: {len(public) - no_coord}곳")

결측치가 있는 컬럼:
latitude       27
longitude      27
tel            26
day_max_fee    12
month_fee      11
add_time        1

좌표 없음: 27곳 / 전체 43곳
지도 표시 가능: 16곳


In [36]:
# 좌표 없는 주차장은 어떤 유형인지 확인
public[public["latitude"].isna()][["parking_name", "lot_type", "address"]].head(10)

,parking_name,lot_type,address
5,신문로 공영주차장(구),노외 주차장,종로구 신문로1가 58-36
9,세운상가밑(구),노상 주차장,종로구 장사동 116-3
10,낙산성곽 버스전용 주차장(구),노상 주차장,종로구 창신동 615-62
11,열린마당(구),노상 주차장,종로구 세종로 111-0 옆
12,종각(구),노상 주차장,종로구 관철동 45-9
13,원서공원 앞(구),노상 주차장,종로구 와룡동 206-0
14,광화문빌딩(구),노상 주차장,종로구 당주동 40-9
15,삼청공원앞(구),노상 주차장,종로구 삼청동 25-11
24,평창동 460-3(구),노상 주차장,종로구 평창동 460-3
25,평창동(구),노상 주차장,종로구 평창동 114-3


## 6. 전국주차장정보표준데이터 (민영 · 부설)

데이터셋: [전국주차장정보표준데이터](https://www.data.go.kr/data/15012896/standard.do)

서울시 데이터에는 **공영만** 있습니다. 민영·부설을 얻으려면 이 표준데이터가 필요합니다.
`주차장구분`(공영/민영), `주차장유형`(노상/노외/**부설**) 컬럼이 핵심입니다.

> CSV를 아직 안 받았다면 아래 셀은 샘플 데이터로 대체 실행됩니다.

In [37]:
std_path = ROOT / "data/raw/national_parking.csv"

if std_path.exists():
    from collectors.standard_parking_api import load_csv, filter_district as std_filter, to_table_schema as std_schema
    std_raw = load_csv(str(std_path))
    std_jongno = std_filter(std_raw, "종로구")
    standard = std_schema(std_jongno)
    print(f"표준데이터에서 종로구 {len(standard)}곳 확보")
else:
    print("⚠ data/raw/national_parking.csv 없음 → 샘플 데이터로 진행합니다.")
    print("  https://www.data.go.kr/data/15012896/standard.do 에서 CSV 다운로드 후 재실행하세요.\n")
    standard = pd.DataFrame([
        {"ext_id": "S1", "parking_name": "세종로공영주차장", "address": "종로구 세종로 80-1",
         "lot_category": "공영", "lot_type": "노외", "latitude": 37.5735, "longitude": 126.9760,
         "capacity": 1260, "base_fee": 430, "base_time": 5, "add_fee": 430, "add_time": 5,
         "weekday_start": 0, "weekday_end": 2400, "source": "standard"},
        {"ext_id": "S2", "parking_name": "인사동 그랑서울", "address": "종로구 인사동",
         "lot_category": "민영", "lot_type": "노외", "latitude": 37.5717, "longitude": 126.9857,
         "capacity": 120, "base_fee": 1000, "base_time": 10, "add_fee": 500, "add_time": 10,
         "weekday_start": 0, "weekday_end": 2400, "source": "standard"},
        {"ext_id": "S3", "parking_name": "광화문D타워", "address": "종로구 청진동",
         "lot_category": "민영", "lot_type": "부설", "latitude": 37.5711, "longitude": 126.9780,
         "capacity": 300, "base_fee": 1200, "base_time": 15, "add_fee": 600, "add_time": 10,
         "weekday_start": 700, "weekday_end": 2200, "source": "standard"},
        {"ext_id": "S4", "parking_name": "낙원상가", "address": "종로구 낙원동",
         "lot_category": "민영", "lot_type": "노외", "latitude": 37.5730, "longitude": 126.9880,
         "capacity": 80, "base_fee": 800, "base_time": 10, "add_fee": 400, "add_time": 10,
         "weekday_start": 600, "weekday_end": 2400, "source": "standard"},
    ])

standard[["parking_name", "lot_category", "lot_type", "capacity"]]

⚠ data/raw/national_parking.csv 없음 → 샘플 데이터로 진행합니다.
  https://www.data.go.kr/data/15012896/standard.do 에서 CSV 다운로드 후 재실행하세요.



,parking_name,lot_category,lot_type,capacity
0,세종로공영주차장,공영,노외,1260
1,인사동 그랑서울,민영,노외,120
2,광화문D타워,민영,부설,300
3,낙원상가,민영,노외,80


In [38]:
print("주차장 구분별:")
print(standard["lot_category"].value_counts().to_string())
print("\n주차장 유형별:")
print(standard["lot_type"].value_counts().to_string())

주차장 구분별:
lot_category
민영    3
공영    1

주차장 유형별:
lot_type
노외    3
부설    1


## 7. 다중 소스 통합 + 중복 제거 ⚠️ 핵심 처리

같은 주차장이 소스마다 **다른 이름**으로 들어있습니다.

- 서울시 공영: `세종로 공영주차장(시)`
- 표준데이터: `세종로공영주차장`

그냥 합치면 지도에 마커가 두 개 찍히고 추천 순위도 왜곡됩니다.

**판정 기준**: 이름 정규화 후 일치 **AND** 좌표 50m 이내

In [39]:
from collectors.merge_parking import normalize_name

samples = ["세종로 공영주차장(시)", "세종로공영주차장", "종묘주차장 공영주차장(시)", "청계2(북2) 공영주차장(시)"]
pd.DataFrame({"원본 이름": samples, "정규화 결과": [normalize_name(s) for s in samples]})

,원본 이름,정규화 결과
0,세종로 공영주차장(시),세종로
1,세종로공영주차장,세종로
2,종묘주차장 공영주차장(시),종묘
3,청계2(북2) 공영주차장(시),청계2


In [40]:
from collectors.merge_parking import dedupe, to_table_schema as merge_schema

public["source"] = "seoul_public"
combined = pd.concat([public, standard], ignore_index=True)
print(f"단순 합계: {len(combined)}행")

deduped = dedupe(combined)
final = merge_schema(deduped)
print(f"중복 제거 후: {len(final)}행")

단순 합계: 47행
중복 제거: 1건 병합 (47 -> 46)
중복 제거 후: 46행


In [41]:
print("최종 구분별 개수:")
print(final["lot_category"].value_counts(dropna=False).to_string())

with_coord = final[["latitude", "longitude"]].notna().all(axis=1).sum()
print(f"\n좌표 보유: {with_coord}/{len(final)}곳")

최종 구분별 개수:
lot_category
공영    43
민영     3

좌표 보유: 19/46곳


In [42]:
# 민영·부설만 확인
final[final["lot_category"].isin(["민영", "부설"])][
    ["parking_name", "lot_category", "lot_type", "capacity", "address"]
]

,parking_name,lot_category,lot_type,capacity,address
43,인사동 그랑서울,민영,노외,120,종로구 인사동
44,광화문D타워,민영,부설,300,종로구 청진동
45,낙원상가,민영,노외,80,종로구 낙원동


## 8. 예상 요금 · 추천 점수 계산

`common/recommend.py`의 로직을 그대로 사용합니다.

**예상 요금** = 기본요금 + ⌈(주차시간 − 기본시간) / 추가단위시간⌉ × 추가요금
(일 최대요금이 있으면 상한 적용)

In [43]:
from common.recommend import estimate_fee

# 세종로 공영주차장 기준: 5분당 430원
for h in [0.5, 1, 2, 4, 8]:
    fee = estimate_fee(h, base_fee=430, base_time=5, add_fee=430, add_time=5)
    print(f"{h:>4}시간 주차 → {fee:>8,}원")

 0.5시간 주차 →    2,580원
   1시간 주차 →    5,160원
   2시간 주차 →   10,320원
   4시간 주차 →   20,640원
   8시간 주차 →   41,280원


In [44]:
from common.geo import haversine_km
from common.recommend import rank_parking_lots
from datetime import datetime

# 광화문광장 기준 1km 이내
CENTER_LAT, CENTER_LNG = 37.5720, 126.9769
RADIUS_KM = 1.0

cand = final.dropna(subset=["latitude", "longitude"]).copy()
cand["distance_km"] = cand.apply(
    lambda r: haversine_km(CENTER_LAT, CENTER_LNG, r["latitude"], r["longitude"]), axis=1
)
cand = cand[cand["distance_km"] <= RADIUS_KM]
print(f"반경 {RADIUS_KM}km 이내: {len(cand)}곳")

반경 1.0km 이내: 7곳


In [45]:
ranked = rank_parking_lots(
    cand, radius_km=RADIUS_KM, hours=2.0,
    w_distance=0.5, w_availability=0.3, w_fee=0.2,
    now=datetime(2026, 7, 27, 14, 0),   # 재현 가능하도록 시각 고정
)

ranked[["parking_name", "lot_category", "distance_km", "estimated_fee", "total_score"]].round(3)

,parking_name,lot_category,distance_km,estimated_fee,total_score
0,광화문D타워,민영,0.139,7800,0.687
1,세종로 공영주차장(시),공영,0.180,10320,0.637
2,서울글로벌센터 공영주차장(시),공영,0.472,16800,0.414
3,인사동 그랑서울,민영,0.776,6500,0.384
4,낙원상가,민영,0.985,5200,0.296
5,관철동 공영주차장(시),공영,0.917,10320,0.269


### 가중치를 바꾸면 순위가 바뀝니다

추천 로직이 실제로 작동하는지 확인하는 가장 좋은 방법입니다.

In [46]:
def top3(w_dist, w_avail, w_fee, label):
    r = rank_parking_lots(cand, RADIUS_KM, 2.0, w_dist, w_avail, w_fee,
                          now=datetime(2026, 7, 27, 14, 0))
    names = " > ".join(r["parking_name"].head(3))
    print(f"{label:14s}: {names}")

top3(1, 0, 0, "거리 우선")
top3(0, 0, 1, "요금 우선")
top3(0.5, 0.3, 0.2, "균형(기본값)")

거리 우선         : 광화문D타워 > 세종로 공영주차장(시) > 서울글로벌센터 공영주차장(시)
요금 우선         : 낙원상가 > 인사동 그랑서울 > 광화문D타워
균형(기본값)       : 광화문D타워 > 세종로 공영주차장(시) > 서울글로벌센터 공영주차장(시)


### 점수 구성 요소 살펴보기

In [47]:
ranked[["parking_name", "score_distance", "score_availability", "score_fee", "total_score"]].round(3)

,parking_name,score_distance,score_availability,score_fee,total_score
0,광화문D타워,0.861,0.5,0.536,0.687
1,세종로 공영주차장(시),0.820,0.5,0.386,0.637
2,서울글로벌센터 공영주차장(시),0.528,0.5,0.000,0.414
3,인사동 그랑서울,0.224,0.5,0.613,0.384
4,낙원상가,0.015,0.5,0.690,0.296
5,관철동 공영주차장(시),0.083,0.5,0.386,0.269


## 9. 최종 저장

DB 적재용 CSV로 저장합니다.

```bash
uv run python loaders/load_to_db.py \
    --csv data/cleaned/parking_lot.csv \
    --table PARKING_LOT --if-exists replace
```

In [48]:
out_path = ROOT / "data/cleaned/parking_lot.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
final.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"저장 완료 → {out_path}")
print(f"{len(final)}행 × {len(final.columns)}컬럼")

저장 완료 → /Users/lsh/Library/Application Support/Claude/local-agent-mode-sessions/afaac2f5-30a2-4216-a837-eb123ec06b0c/02bcc11c-9e38-4f9b-92ea-d2845619930d/local_ecbc54ba-1f5e-4442-a756-62b43843cf51/outputs/team-project-scaffold-v7/data/cleaned/parking_lot.csv
46행 × 24컬럼


## 정리

| 단계 | 결과 |
|---|---|
| 서울시 원본 | 2,189행 (서울 전체) |
| 종로구 필터 | 210행 → 고유 43곳 |
| 좌표 중복 병합 | 43행 (주차장당 1행) |
| 표준데이터 추가 | 민영 · 부설 확보 |
| 소스 통합 + 중복 제거 | 최종 통합 테이블 |

### 남은 과제

1. `data/raw/national_parking.csv` 실제 다운로드 → 종로구 민영·부설 실데이터 확보
2. 좌표 없는 27곳 → 카카오 Geocoding으로 보완 가능 (REST 키 필요)
3. 실시간 주차대수 API(OA-21709) 인증키 발급 → 여유 점수 활성화
4. `park.ijongno.co.kr` 거주자우선주차 크롤링 (Selenium 필요)
